# Task 1–4 实验结果总览

本 notebook 汇集 `thesis/report-assets/tables/` 中五张报告表的**全部已整理行**，并解释各指标的含义与可比较范围。数据取自项目已有的冻结评估或经核验的汇总报告；点击表格里的 `source_path` 可以回到原始证据说明。

本文件是写作与复核材料。Notebook 中的表格是 CSV 的快照；运行全部单元格可读取当前 CSV 重新显示。它没有覆盖仓库里每个历史 checkpoint 和所有 `runs/` 逐局日志；模型清单中的证据缺口会原样显示。最终课程 PDF 仍由团队成员核验、改写并单独提交。

## 0. 如何运行与数据来源

从仓库根目录启动 JupyterLab，打开 `thesis/report-assets/task1_to_task4_results.ipynb`，选择 Python 3 kernel 后运行全部单元格。读取逻辑会自动寻找仓库根目录或 notebook 所在目录。所需依赖是 Jupyter 的 `IPython` 与 Python 标准库；原仓库 `environment.yml` 已声明 JupyterLab。

五张 CSV 分别是模型清单、Task 1/2、Task 3 配对、Task 4 尝试与最终候选。四张图来自 `figures/output/`，图 3、图 4 的源数值在 `figures/data/`。这两组数据与表格共享相同证据边界。

In [ ]:
from pathlib import Path
import csv
from html import escape
from IPython.display import HTML, display

def locate_assets():
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        for candidate in (base, base / "thesis" / "report-assets"):
            if (candidate / "tables" / "table1_model_inventory.csv").is_file():
                return candidate
    raise FileNotFoundError("未找到 thesis/report-assets；请从仓库或 notebook 目录启动 Jupyter。")

ASSETS = locate_assets()
TABLE_FILES = {
    1: "table1_model_inventory.csv",
    2: "table2_task1_task2.csv",
    3: "table3_task3_paired.csv",
    4: "table4_task4_attempts.csv",
    5: "table5_die_hardest.csv",
}

def read_rows(filename):
    with (ASSETS / "tables" / filename).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def show_rows(rows, *, columns=None, label=""):
    if not rows:
        display(HTML("<p>没有符合条件的数据行。</p>"))
        return
    columns = columns or list(rows[0])
    th = "".join(f'<th style="position:sticky;top:0;background:#eef2f7;border:1px solid #ced6e0;padding:6px;text-align:left;">{escape(key)}</th>' for key in columns)
    trs = []
    for row in rows:
        cells = []
        for key in columns:
            value = str(row.get(key, ""))
            shown = escape(value)
            if key == "source_path":
                shown = f'<a href="../../{shown}">{shown}</a>'
            cells.append(f'<td style="border:1px solid #d8dfe8;padding:6px;white-space:nowrap;">{shown}</td>')
        trs.append("<tr>" + "".join(cells) + "</tr>")
    title = f"<p><b>{escape(label)}</b> · {len(rows)} rows</p>" if label else f"<p>{len(rows)} rows</p>"
    display(HTML(title + '<div style="max-height:520px;overflow:auto;"><table style="border-collapse:collapse;font-size:12px;"><thead><tr>' + th + '</tr></thead><tbody>' + "".join(trs) + '</tbody></table></div>'))

results = {number: read_rows(filename) for number, filename in TABLE_FILES.items()}
print("已加载五张表：", {number: len(rows) for number, rows in results.items()})


## 1. 评估口径与指标解释

报告中的“局”指一次冻结策略对局；`training_seed` 是一次独立训练初始化，`environment_seed` 是测试世界种子。同一个 checkpoint 在 100 个世界中评估是 **100 次环境重复**，不能写成 100 次独立训练。开发集用于选择 checkpoint；独立确认或主验证应在候选固定后运行。不同场景、对手、世界集合、模型合同或评估阶段的均值不能当作配对差。

| Metric | 计算方式与读法 | 适用范围 |
|---|---|---|
| `mean_score` | 官方每局得分均值。游戏中每枚金币 +1，每次击杀 +5；`mean_score` 不能与 shaped training reward 混用。 | Task 1–4 |
| `mean_coins` | 每局收集金币数的均值。Task 1 通常以 50 枚为满额；Task 2 以 9 枚为满额。Task 3/4 有击杀时，它不等于 `mean_score`。 | Task 1–4 |
| `all_coins_rate` | 收齐该任务全部金币的局数 ÷ 评估局数。例如 `0.96` 是 96%，若评估 100 局就是 96 局。 | Task 1–2 |
| `mean_crates` | 每局炸毁箱子数的均值。炸箱体现通路与隐藏金币探索，但箱子本身不直接计入官方得分。 | Task 2–4 |
| `mean_kills` | 每局击杀对手数均值。Task 3 主验证父子差 `−0.01`，其 95% CI 跨 0；不能据此声称击杀改善。 | Task 3–4 |
| `first_place_rate` | 第一名局数占比；必须核对是否包括并列。区分 `exclusive_first_rate`（独占第一）和 `tied_first_rate`（并列第一）。 | Task 3–4 |
| `survival_rate`、`suicide_rate` | 前者是活到局末的比例；后者是自己炸死自己的局比例。二者**不是互补量**：死亡还可能有别的原因。 | Task 2–4 |
| `bomb_survival_rate` | 在源协议的已结算炸弹集合中放弹后存活的比例。分母是炸弹事件，不是对局数。 | Task 2–4 |
| `long_wait_rate`、`long_ping_pong_rate` | 出现长时间 WAIT 或长往返问题的局比例；阈值按原实验诊断定义，不同定义不能直接合并。 | Task 2 |
| `invalid_action_rate` | 无效动作数 ÷ 动作决策数。项目内部工程门槛通常为不超过 1%。 | Task 1–4 |
| `act_p95_ms`、`act_max_ms` | 完整 `act` 的第 95 百分位和最大耗时，单位毫秒。官方动作预算为 500 ms；不同硬件的时延不可当作模型效率因果比较。 | Task 1–4 |
| `delta` 与 `ci95_low/high` | 子模型减父模型的**同世界配对差**与 95% bootstrap 区间。比例差通常用百分点（pp）描述；例如 0.12 = 12 pp。 | Task 3 配对 |

`not_reported` 表示来源未给出该数值；`not_applicable` 表示该场景不适用；`unrun` 表示该阶段没有运行。这些状态都不能填成 0。详细定义见 [指标字典](tables/task-metric-dictionary.md) 和 [统一协议](plan/experiment-protocol.md)。

## 2. 模型与证据覆盖（Table 1）

这里的每一行是模型族或代表候选，不等于每个变体都完成了四个任务。状态列保留历史基线、验证结果和证据缺口。

In [2]:
show_rows(results[1], label="Table 1 · Model inventory")

model_family,input_representation,learner,reward_or_training_contract,covered_tasks,status,evidence_status,source_path
Tabular Q-learning,discrete-v1/discrete-q-v2,Q-learning,exploratory Task 1,1,archived baseline,verified_summary,thesis/REPORT_OUTLINE.md
Double Q(lambda),84-D continuous-v2 tile coding,Watkins Double Q(lambda),r20 + survival-mask-v1/all,1-2,Task 1 independently confirmed; Task 2 below target,verified_summary,docs/research/q-cnn-experiment-report.md
CNN Double DQN,17-channel board/path/history,Double DQN,r3/r5 experiments,1,early baseline and input-bug history,verified_summary,docs/research/q-cnn-experiment-report.md
Distilled CNN Double DQN,17-channel CNN + frozen teacher during training,Double DQN,Task 1 teacher KL then TD fine-tuning,1-2,Task 1 confirmed; Task 2 partial transfer,verified_summary,docs/research/q-cnn-experiment-report.md
Continuous Double DQN,70/78/84-D continuous,Double DQN,r7 family + survival-mask versions,1-4,central Task 2-4 lineage,verified_summary,thesis/REPORT_OUTLINE.md
Phase Double DQN,117-D phase features,Double DQN,Task 3 phase contracts,3,retention/safety gate failures,verified_summary,thesis/REPORT_OUTLINE.md
Rainbow-lite,continuous features + selected Rainbow modules,Double DQN variant,experiment-specific,1-2,implemented; admissible aggregate result not audited,evidence_gap,thesis/REPORT_OUTLINE.md
Task 4 specialists,84-D continuous,Double DQN,E1/E2/E3 and frozen C/S contracts,4,bounded experiments; no stable score gain,verified_raw,experiments/results/task4_exploration_20260919/report.md
Die Hardest,continuous-v2,Double DQN,r7_safe_credit_sparse + survival-mask-v9,4,frozen submission candidate,verified_summary,agent_code/die_hardest/README.md


## 3. Task 1：金币导航（Table 2）

Task 1 主要看 `mean_coins` 与 `all_coins_rate`；平均金币高但捡满率低，说明模型可能反复漏掉最后几枚。蒸馏 CNN 的 reserved 100 局为 96% 捡满、49.84 枚；Double Q(λ) 的独立 100 局确认为 96%、48.95 枚。两条结果的评估集合与训练合同不同，可并列展示，不能把 0.89 枚写作同协议配对提升。早期 path CNN 的输入错误与后续修复版需要分开解释。

下表展示 Table 2 的所有 Task 1 行；字段里的 `below 50` / `below 500` 只表示报告提供了门槛结论，没有精确毫秒数。

In [3]:
show_rows([row for row in results[2] if row["task"] == "1"], label="Task 1 frozen results")

candidate,model_family,task,protocol_id,episodes_or_worlds,mean_coins,all_coins_rate,mean_crates,long_wait_rate,long_ping_pong_rate,suicide_rate,bomb_survival_rate,act_p95_ms,act_max_ms,result_status,evidence_status,source_path
path CNN r3,CNN Double DQN,1,feature-v2 r3 frozen best,100,13.35,0.00,not_applicable,not_reported,not_reported,not_reported,not_applicable,not_reported,not_reported,failed,verified_summary,experiments/agent_variants/cnn_path_double_dqn_agent/EXPERIMENT_LOG.md
path CNN r5,CNN Double DQN,1,feature-v2 frozen best,100,41.78,0.17,not_applicable,not_reported,not_reported,not_reported,not_applicable,not_reported,not_reported,below target,verified_summary,docs/research/q-cnn-experiment-report.md
distilled CNN global,Distilled CNN Double DQN,1,reserved frozen confirmation,100,49.84,0.96,not_applicable,not_reported,not_reported,not_reported,not_applicable,8.39,15.55,passed Task 1,verified_summary,docs/research/q-cnn-experiment-report.md
optimized Double Q(lambda),Double Q(lambda),1,independent frozen confirmation,100,48.95,0.96,not_applicable,not_reported,not_reported,not_reported,not_applicable,below 50,below 500,passed Task 1,verified_summary,docs/research/q-cnn-experiment-report.md


## 4. Task 2：隐藏金币、炸箱与循环（Table 2）

Task 2 必须同时看金币、炸箱、安全、WAIT/往返。Q(λ) r20 的平均 3.95/9 枚、53.45 箱，长 WAIT 局率 30%、长往返局率 75%；安全放弹没有解决后期目标切换。蒸馏 CNN D02 的主验证平均 2.60/9 枚、44.15 箱，仍未达到全金币目标。Q 变体的 20 局开发评估与 CNN 的 100 局主验证**不是同口径竞赛排名**。

与 Task 2 内部门槛对照时，可关注平均金币至少 6/9、平均炸箱至少 60、自杀率不超过 5%、长 WAIT 不超过 10% 和长往返不超过 5%。其他完整门槛以 [全局协议](../../experiments/CURRENT_TRAINING_EVALUATION_PARAMETERS.md) 为准。

In [4]:
show_rows([row for row in results[2] if row["task"] == "2"], label="Task 2 frozen results")

candidate,model_family,task,protocol_id,episodes_or_worlds,mean_coins,all_coins_rate,mean_crates,long_wait_rate,long_ping_pong_rate,suicide_rate,bomb_survival_rate,act_p95_ms,act_max_ms,result_status,evidence_status,source_path
Q(lambda) r20 175.2k,Double Q(lambda),2,frozen development checkpoint,20,3.95,not_reported,53.45,0.30,0.75,0.00,1.00,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md
Q crate quantized,Double Q(lambda),2,frozen development checkpoint,20,1.35,not_reported,not_reported,above threshold,above threshold,not_reported,not_reported,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md
Q history input,Double Q(lambda),2,frozen development checkpoint,20,1.65,not_reported,29.70,above threshold,0.85,not_reported,not_reported,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md
Q continuous-v4 r12,Double Q(lambda),2,frozen development checkpoint,20,0.95,not_reported,18.40,0.00,0.35,not_reported,not_reported,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md
Q grouped tiles,Double Q(lambda),2,frozen development checkpoint,20,0.00,not_reported,0.00,0.05,0.65,not_reported,not_reported,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md
Q team demo 50k,Double Q(lambda),2,frozen development checkpoint,20,3.75,not_reported,56.90,0.65,0.75,not_reported,not_reported,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md
distilled CNN D02,Distilled CNN Double DQN,2,main validation,100,2.60,0.00,44.15,not_reported,not_reported,0.00,not_reported,not_reported,not_reported,below Task 2 target,verified_summary,docs/research/q-cnn-experiment-report.md


## 5. Task 3：父子配对与置信区间（Table 3）

三条训练 seed 在确认集上单列；唯一候选 seed 22/c150 的主验证在另 100 个世界中完成。主验证的 `mean_score` 为 5.91 → 7.14，配对差 +1.23，95% CI [0.27, 2.14]。第一名占比 41% → 53%，差 12 pp，但当前汇总表没有该合并指标的配对 CI。击杀为 0.45 → 0.44，差 −0.01，CI [−0.16, 0.14]；报告结论应限制在得分和第一名占比改善。

In [5]:
show_rows(results[3], label="Table 3 · Task 3 paired evidence")

stage,training_seed,task,metric,parent,child,delta,ci95_low,ci95_high,episodes_or_worlds,evidence_status,source_path
main_validation,22,3,mean_score,5.91,7.14,1.23,0.27,2.14,100,verified_raw,docs/research/task3-counter-validation-results.md
main_validation,22,3,mean_coins,3.66,4.94,1.28,0.87,1.70,100,verified_raw,docs/research/task3-counter-validation-results.md
main_validation,22,3,mean_crates,43.78,58.97,15.19,12.17,18.16,100,verified_raw,docs/research/task3-counter-validation-results.md
main_validation,22,3,mean_kills,0.45,0.44,-0.01,-0.16,0.14,100,verified_raw,docs/research/task3-counter-validation-results.md
main_validation,22,3,first_place_rate,0.41,0.53,0.12,not_reported,not_reported,100,verified_summary,docs/research/task3-counter-validation-results.md
confirmation,11,3,mean_score,4.92,6.84,1.92,1.06,2.80,100,verified_raw,docs/research/task3-counter-validation-results.md
confirmation,22,3,mean_score,5.88,7.11,1.23,0.16,2.28,100,verified_raw,docs/research/task3-counter-validation-results.md
confirmation,33,3,mean_score,4.75,7.31,2.56,1.59,3.54,100,verified_raw,docs/research/task3-counter-validation-results.md


## 6. Task 4：专项尝试与未运行阶段（Table 4）

E1/E2/E3 的末端冻结结果、Frozen C/S 和未完成的 score campaign 属于不同协议。可以逐项讨论设想、到达的阶段与停止原因，不应合并均值或按表中得分直接排序。表中的 `unrun` 保留为未运行；空值不是失败得分 0。

In [6]:
show_rows(results[4], label="Table 4 · Task 4 attempts")

attempt,protocol_id,training_seed,stage_or_actions,mean_score,mean_coins,mean_kills,first_place_rate,suicide_rate,bomb_survival_rate,act_p95_ms,act_max_ms,outcome,stop_cause,evidence_status,source_path
E1,task4_exploration_E1,11,100390 actions,4.06,2.61,0.29,0.44,0.00,1.00,19.98,229.09,no stable score gain,bounded experiment ended,verified_raw,experiments/results/task4_exploration_20260919/report.md
E1,task4_exploration_E1,22,100002 actions,4.10,2.80,0.26,0.43,0.00,1.00,19.12,165.31,no stable score gain,bounded experiment ended,verified_raw,experiments/results/task4_exploration_20260919/report.md
E1,task4_exploration_E1,33,100178 actions,3.27,2.47,0.16,0.34,0.00,1.00,22.25,151.37,no stable score gain,bounded experiment ended,verified_raw,experiments/results/task4_exploration_20260919/report.md
E2,task4_exploration_E2,22,100288 actions,2.34,1.89,0.09,0.25,0.00,1.00,19.61,175.61,worse observed endpoint,bounded experiment ended,verified_raw,experiments/results/task4_exploration_20260919/report.md
E3,task4_exploration_E3,22,100130 actions,2.79,2.04,0.15,0.26,0.00,1.00,18.92,155.00,no stable score gain,bounded experiment ended,verified_raw,experiments/results/task4_exploration_20260919/report.md
Frozen C,task4_frozen_C,22,60055 actions,4.055,2.755,0.260,0.425,0.00,1.00,19.44,122.13,initial screen not promoted,no observed advantage,not_comparable,experiments/results/task4_frozen_20260919/report.md
Frozen S,task4_frozen_S,22,60345 actions,4.035,2.660,0.275,0.475,0.00,1.00,19.30,155.22,initial screen not promoted,no observed advantage,not_comparable,experiments/results/task4_frozen_20260919/report.md
Score L/P/K,task4_score_campaign,not_reported,partial campaign,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,unrun final comparison,worker failure / interruption,unrun,experiments/results/task4_score_20260920/report.md
Shared-parent engineering,task4_shared_parent,not_reported,admission only,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,unrun formal attempts,engineering admission failure,unrun,docs/research/task4-frozen-results.md


## 7. 最终候选 Die Hardest：性能与打包证据（Table 5）

原 B33 seed33/c200 对三个 `rule_based_agent` 的历史 1,000-world 评估为均分 4.231、含并列第一 50.2%、独占第一 37.8%、存活 94.7%。重命名后的 `die_hardest` 完成 180 条 trace 与六局配对行为等价性验证；这六局不构成新的性能估计。打包检查的 P95/max 为 14.826/44.601 ms，峰值 RSS 约 319.645 MiB；不能移用到历史 1,000-world 性能行。已知 27155 安全反例仍存在，完整安全认证、Docker 和官方硬件验证没有完成。

In [7]:
show_rows(results[5], label="Table 5 · Final candidate and package checks")

record_type,candidate,protocol,episodes_or_worlds,mean_score,first_place_rate,exclusive_first_rate,survival_rate,act_p95_ms,act_max_ms,peak_rss_mib,known_limitations,evidence_status,source_path
historical_benchmark,Task4 B seed33/c200,1000 worlds versus three rule_based agents,1000,4.231,0.502,0.378,0.947,not_reported,not_reported,not_reported,not a rerun after rename; historical safety counterexample remains,verified_summary,agent_code/die_hardest/README.md
package_equivalence,die_hardest,6 paired games plus trace equivalence,6,not_reported,not_reported,not_reported,not_reported,14.826,44.601,319.645,packaging only; Docker and official hardware untested; full safety not approved,packaging_verified,docs/research/die-hardest-submission/verification.json
trace_equivalence,die_hardest,renamed package against source candidate,180 trace rows,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,not_reported,27155 safety counterexample reproduces at step 77,packaging_verified,docs/research/die-hardest-submission/verification.json


## 8. 图表与图表数据

图 1–2 是流程示意；图 3 按任务分面；图 4 仅绘制同时有 score 和 P95 的专项评估点。B33 的历史性能没有对应 P95，所以只作文字注记；打包检查没有新的 score，也只作文字注记。这样不会把两个不同评估拼成一个坐标点。

![Figure 1：课程流程](figures/output/figure_01_curriculum_pipeline.png)

![Figure 2：推理流程](figures/output/figure_02_inference_pipeline.png)

![Figure 3：分任务能力](figures/output/figure_03_capability_progression.png)

![Figure 4：Task 4 得分与延迟](figures/output/figure_04_task4_tradeoff.png)

图表的数据、图注与生成命令见 [data-manifest](figures/data-manifest.md)。

## 9. 完整性检查与可复核边界

下面的单元格检查五张表是否都非空、每行是否有现存的源文件，以及 `unrun` 行的性能指标是否保留缺失标记。它只检查追溯结构；数值本身仍需与来源报告核对。表格中的 `verified_summary` 表示来自已核验汇总，`verified_raw` 表示项目数据资产把它登记为原始/冻结记录，`not_comparable` 表示协议不同，`packaging_verified` 只限包验证。

In [ ]:
ALLOWED_STATUS = {"verified_raw", "verified_summary", "not_comparable", "unrun", "packaging_verified", "evidence_gap"}
REPO = ASSETS.parents[1]
for table_no, rows in results.items():
    assert rows, f"Table {table_no} is empty"
    for row in rows:
        assert row.get("evidence_status") in ALLOWED_STATUS, (table_no, row)
        source = row.get("source_path")
        assert source and (REPO / source).exists(), (table_no, source)
        if row["evidence_status"] == "unrun":
            for field in ("mean_score", "mean_coins", "mean_kills"):
                if field in row:
                    assert row[field] == "not_reported", (table_no, field, row[field])
print("完整性检查通过：", sum(map(len, results.values())), "条表格记录，5 张表，全部有来源与证据状态。")


## 10. 报告写作提示

- Task 1 的独立确认支持“该模型已学会导航”，但不能用两个不同确认集的均值差声称 CNN 或 Q(λ) 架构因果优越。
- Task 2 的负面结果应同时保留金币、炸箱、循环和安全指标，避免仅凭炸弹数或训练 reward 判断成功。
- Task 3 的主验证可写 score 与第一名占比的观察改善；击杀差的 CI 跨 0。
- Task 4 的历史 benchmark、专项冻结结果和打包等价测试分别引用；缺失的评估保持 `not_reported` 或 `unrun`。
- Notebook 是可复核写作素材。最终 PDF 的正文、作者署名、实验局限与引用应由团队成员核验并改写。